1. Kubernetes Setup: Set up a local Kubernetes cluster using Minikube or a
managed Kubernetes service ? (e.g., GKE, EKS).


### 1. Authenticate with Google Cloud
You'll need to authenticate your Colab session to interact with your GCP resources.

In [1]:
from google.colab import auth
auth.authenticate_user()

### 2. Configure gcloud
Replace `'your-project-id'` with your actual GCP Project ID and choose a zone (e.g., `'us-central1-a'`).

In [2]:
project_id = 'your-project-id' # @param {type:"string"}
zone = 'us-central1-a' # @param {type:"string"}

!gcloud config set project {project_id}
!gcloud config set compute/zone {zone}

Are you sure you wish to set property [core/project] to your-project-id?

Do you want to continue (Y/n)?  y

Updated property [core/project].


To take a quick anonymous survey, run:
  $ gcloud survey

API [compute.googleapis.com] not enabled on project [your-project-id]. Would you
 like to enable and retry (this will take a few minutes)? (y/N)?  y

Enabling service [compute.googleapis.com] on project [your-project-id]...
ERROR: (gcloud.config.set) PERMISSION_DENIED: Permission denied to enable service [compute.googleapis.com]
Help Token: AerXPhUva6PNzV_B-4dJnJz_Zeidh1w5th-pEW0msUf9iSRXaHjvzXpaS2oFIzjW6eqz1l9b9Dpc9Z6CZBfYY6mY74Nq55_v-cwF1cxe89L6-ZVn. This command is authenticated as pavan201103@gmail.com which is the active account specified by the [core/account] property
- '@type': type.googleapis.com/google.rpc.PreconditionFailure
  violations:
  - subject: '110002'
    type: googleapis.com
- '@type': type.googleapis.com/google.rpc.ErrorInfo
  domain: serviceusage.googleapis.com
  re

### 3. Enable Container API and Create Cluster
This command enables the GKE API and creates a small 3-node cluster named `my-cluster`.

In [3]:
# Enable the GKE API
!gcloud services enable container.googleapis.com

# Create the cluster
!gcloud container clusters create my-cluster \
    --num-nodes=3 \
    --machine-type=e2-medium

ERROR: (gcloud.services.enable) PERMISSION_DENIED: Permission denied to enable service [container.googleapis.com]
Help Token: AerXPhXGiULA65C-t-ldIzn50M2hQlKPWuNNoQD7zkms0_mRa9jnfnKlRv54kj5sNw3Aflcq03hWIZrWM1ElB8pLPk3POIJUagNU9WzQrEqjZip1. This command is authenticated as pavan201103@gmail.com which is the active account specified by the [core/account] property
- '@type': type.googleapis.com/google.rpc.PreconditionFailure
  violations:
  - subject: '110002'
    type: googleapis.com
- '@type': type.googleapis.com/google.rpc.ErrorInfo
  domain: serviceusage.googleapis.com
  reason: AUTH_PERMISSION_DENIED
client [kubectl]. To install, run
  $ gcloud components install kubectl

ERROR: (gcloud.container.clusters.create) One of [--location, --zone, --region] must be supplied.


### 4. Verify the Cluster
Once the cluster is created, we'll fetch the credentials and check the nodes using `kubectl`.

In [4]:
!gcloud container clusters get-credentials my-cluster --zone {zone}
!kubectl get nodes

client [kubectl]. To install, run
  $ gcloud components install kubectl

Fetching cluster endpoint and auth data.
ERROR: (gcloud.container.clusters.get-credentials) ResponseError: code=403, message=Kubernetes Engine API has not been used in project your-project-id before or it is disabled. Enable it by visiting https://console.developers.google.com/apis/api/container.googleapis.com/overview?project=your-project-id then retry. If you enabled this API recently, wait a few minutes for the action to propagate to our systems and retry. This command is authenticated as pavan201103@gmail.com which is the active account specified by the [core/account] property.
/bin/bash: line 1: kubectl: command not found


2. Application Deployment: Deploy a simple application to your Kubernetes
cluster?


In [8]:
# Install kubectl using apt
!apt-get update && apt-get install -y apt-transport-https ca-certificates curl
!curl -fsSL https://pkgs.k8s.io/core:/stable:/v1.28/deb/Release.key | sudo gpg --dearmor -o /etc/apt/keyrings/kubernetes-apt-keyring.gpg
!echo 'deb [signed-by=/etc/apt/keyrings/kubernetes-apt-keyring.gpg] https://pkgs.k8s.io/core:/stable:/v1.28/deb/ /' | sudo tee /etc/apt/sources.list.d/kubernetes.list
!apt-get update
!apt-get install -y kubectl

# Verify installation
!kubectl version --client

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,852 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,929 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,786 kB]
Get:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Ge

In [7]:
import yaml

# Define the Deployment
deployment = {
    'apiVersion': 'apps/v1',
    'kind': 'Deployment',
    'metadata': {'name': 'nginx-deployment'},
    'spec': {
        'replicas': 2,
        'selector': {'matchLabels': {'app': 'nginx'}},
        'template': {
            'metadata': {'labels': {'app': 'nginx'}},
            'spec': {
                'containers': [{
                    'name': 'nginx',
                    'image': 'nginx:latest',
                    'ports': [{'containerPort': 80}]
                }]
            }
        }
    }
}

# Define the Service
service = {
    'apiVersion': 'v1',
    'kind': 'Service',
    'metadata': {'name': 'nginx-service'},
    'spec': {
        'selector': {'app': 'nginx'},
        'ports': [{'protocol': 'TCP', 'port': 80, 'targetPort': 80}],
        'type': 'LoadBalancer'
    }
}

# Write to a file
with open('deployment.yaml', 'w') as f:
    yaml.dump_all([deployment, service], f)

print('Generated deployment.yaml')

# Apply to Kubernetes
!kubectl apply -f deployment.yaml

# Wait and check status
!kubectl get pods
!kubectl get service nginx-service

Generated deployment.yaml
/bin/bash: line 1: kubectl: command not found
/bin/bash: line 1: kubectl: command not found
/bin/bash: line 1: kubectl: command not found


Resource Management: Practice managing Kubernetes resources like
Pods, Services, and Deployments?

### 3. Resource Management Practice
Now that you have a deployment file, let's learn how to manage these resources manually.

#### List Resources
Check the status of your Pods, Deployments, and Services.

In [9]:
# List all pods in the default namespace
!kubectl get pods

# List deployments
!kubectl get deployments

# List services to find the External IP
!kubectl get services

E0313 11:51:41.429644    7854 memcache.go:265] couldn't get current server API group list: the server could not find the requested resource
E0313 11:51:41.441643    7854 memcache.go:265] couldn't get current server API group list: the server could not find the requested resource
E0313 11:51:41.450984    7854 memcache.go:265] couldn't get current server API group list: the server could not find the requested resource
E0313 11:51:41.461331    7854 memcache.go:265] couldn't get current server API group list: the server could not find the requested resource
E0313 11:51:41.469551    7854 memcache.go:265] couldn't get current server API group list: the server could not find the requested resource
Error from server (NotFound): the server could not find the requested resource
E0313 11:51:41.612372    7858 memcache.go:265] couldn't get current server API group list: the server could not find the requested resource
E0313 11:51:41.626428    7858 memcache.go:265] couldn't get current server API gr

#### Scale a Deployment
Increase the number of running Nginx instances from 2 to 5.

In [10]:
!kubectl scale deployment nginx-deployment --replicas=5

# Watch the pods being created
!kubectl get pods

E0313 11:51:42.668545    7872 memcache.go:265] couldn't get current server API group list: the server could not find the requested resource
E0313 11:51:42.680513    7872 memcache.go:265] couldn't get current server API group list: the server could not find the requested resource
E0313 11:51:42.690982    7872 memcache.go:265] couldn't get current server API group list: the server could not find the requested resource
E0313 11:51:42.701898    7872 memcache.go:265] couldn't get current server API group list: the server could not find the requested resource
E0313 11:51:42.712545    7872 memcache.go:265] couldn't get current server API group list: the server could not find the requested resource
Error from server (NotFound): the server could not find the requested resource
E0313 11:51:42.874104    7877 memcache.go:265] couldn't get current server API group list: the server could not find the requested resource
E0313 11:51:42.884169    7877 memcache.go:265] couldn't get current server API gr

#### Inspect a Resource
Get detailed information about a specific pod (useful for debugging).

In [13]:
# Get the name of the first pod using a subshell and assign to variable
pod_name = !kubectl get pods -o jsonpath='{.items[0].metadata.name}'

# Check if a pod name was found and describe it
if pod_name and "Error" not in pod_name[0]:
    !kubectl describe pod {pod_name[0]}
else:
    print("No pods found to describe.")

/bin/bash: -c: line 1: unexpected EOF while looking for matching `''
/bin/bash: -c: line 2: syntax error: unexpected end of file


#### Clean Up
Delete the resources we created to avoid ongoing costs.

In [12]:
!kubectl delete -f deployment.yaml

E0313 11:51:44.864385    7895 memcache.go:265] couldn't get current server API group list: the server could not find the requested resource
E0313 11:51:44.876948    7895 memcache.go:265] couldn't get current server API group list: the server could not find the requested resource
unable to recognize "deployment.yaml": the server could not find the requested resource
unable to recognize "deployment.yaml": the server could not find the requested resource


4. Helm Charts: Use Helm to package and deploy applications on
Kubernetes?


In [16]:
# Install Helm
!curl https://raw.githubusercontent.com/helm/helm/main/scripts/get-helm-3 | bash

# Verify installation
!helm version

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 11929  100 11929    0     0   110k      0 --:--:-- --:--:-- --:--:--  110k
Verifying checksum... Done.
Preparing to install helm into /usr/local/bin
helm installed into /usr/local/bin/helm
version.BuildInfo{Version:"v3.20.1", GitCommit:"a2369ca71c0ef633bf6e4fccd66d634eb379b371", GitTreeState:"clean", GoVersion:"go1.25.8"}


In [14]:
# Create a new chart called my-app
!helm create my-app

# Install the chart
!helm install my-release ./my-app

/bin/bash: line 1: helm: command not found
/bin/bash: line 1: helm: command not found


In [15]:
# List active releases
!helm list

# Uninstall the release to clean up
!helm uninstall my-release

/bin/bash: line 1: helm: command not found
/bin/bash: line 1: helm: command not found
